In [1]:
import openai
from openai import OpenAI

from tqdm.auto import tqdm
import time
import os
import sys


with open("../../../openai_api_key.sh", "r") as f:
    line = f.readline()
    env_var_name, api_key = line.split("=")

/home/users2/vaethdk/.virtualenvs/cts_al/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_activelearning/conversational-tree-search/generation/reimburse/gpt4o


In [3]:
from data.dataset import ReimburseGraphDataset, StandardGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

In [4]:

reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: True
- depth: 20  - degree: 13
- answers: 312
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  9


In [5]:
client = OpenAI(api_key=api_key)

In [ ]:
system = """You are a helpful assistant creating a list of FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Remove some information, especially nouns and named entities, between generated questions.
Use casual language.
Output only the generated paraphrases, separating each paraphrase with a <br> tag."""

NUM_QUESTIONS = 100
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 15000
generated_data = {}


user_text = """Generate 100 FAQ-style questions from the fact: "In the US, you are entitled to 30$ per day, minus any free meals which you choose to decline."""
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": user_text},
]

In [9]:
def parse_output(result):
    uniques = set()
    results = []
    duplicates = 0
    for split in result.split("<br>"):
        cleaned = split.strip().strip("\n")
        if len(cleaned.split(" ")) > 25:
            print("LONG:", cleaned)
        else:
            if not cleaned.lower() in uniques:
                results.append(cleaned.strip())
                uniques.add(cleaned.lower())
            else:
                duplicates += 1
    print(" - duplicates:", duplicates)
    return results

In [19]:

system = """You are generating semantically similar paraphrases for a given response inside <response> tags to some question inside a <question> tag. 
The generated response paraphrases should be human-like and short, using frequently used words and phrases only.
The generated response paraphrases should also still be a plausible answer to the question.
Output only the generated paraphrases, separating each paraphrase with a <br> tag."""

def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} paraphrases for the response <response>{answer_text}</response> to the question <question>{node_text}</question>."""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, answer_text: str, num_paraphrases: int, temperature: float, seed: int):
    output = client.chat.completions.create(
        model="gpt-4o",
        messages=api_prompt(prompt(node_text, answer_text, num_paraphrases)),
        temperature=temperature,
        stream=False,
        seed=seed
    )
    return output.choices[0].message.content

In [21]:
from data.dataset import NodeType, Question
from collections import defaultdict

SEED = 42
TEMPERATURE = 0.7
NUM_PARAPHRASES = 25


generated = defaultdict(lambda: set())

num_generated = 0

for idx, node in tqdm(enumerate(reimburse_human_data.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        response = api_completion(node.text, answer.text, NUM_PARAPHRASES, TEMPERATURE, SEED)
        answers = parse_output(response)

        generated[answer.key] = generated[answer.key].union(answers)

        num_generated += len(answers)

        # print("ANSWER", answer.text)
        # print(answers)
        # break

0it [00:00, ?it/s]

 - duplicates: 0
 - duplicates: 0


1it [00:12, 12.34s/it]

 - duplicates: 0
 - duplicates: 0
 - duplicates: 0
 - duplicates: 1
 - duplicates: 0
 - duplicates: 0


2it [00:29, 15.28s/it]

...

 - duplicates: 0


21it [04:32, 11.18s/it]

 - duplicates: 0
 - duplicates: 0
 - duplicates: 0


22it [04:40, 12.74s/it]

 - duplicates: 0


In [22]:
import json
with open("../../../resources/en/reimburse/generated/gpt4o/train_answers_v2.json", "w") as f:
    formatted = {}
    for answer_key in generated:
        formatted[answer_key] = list(generated[answer_key])
    json.dump(formatted, f)
